# Step 2.1 — Missing Values Analysis

Missing values are not automatically errors. Before applying any imputation technique, we need to understand why values are missing.

The mechanism behind missingness affects the correct preprocessing strategy:

## MCAR (Missing Completely At Random)

Missingness has no relationship with observed or unobserved data.

Example:
- A sensor randomly fails.

If data is MCAR, removing rows may be acceptable if the missing percentage is small.

---

## MAR (Missing At Random)

Missingness depends on other observed variables.

Example:
- Income is missing more often for certain age groups or employment categories.

The missing value itself is not random, but other features can explain the missingness.

Possible approaches:
- KNN imputation
- Regression imputation
- MICE
- Adding missing indicators

---

## MNAR (Missing Not At Random)

Missingness depends on the missing value itself.

Example:
- High-income people avoid reporting income.

The missingness carries information.

Possible approaches:
- Missing indicator features
- Domain-based strategies
- Advanced modeling approaches

---

## Important Rule

No imputation method will be applied without evidence from EDA.

For every missing feature we will:

1. Measure missing percentage.
2. Understand possible missing mechanism.
3. Evaluate impact.
4. Select an appropriate treatment.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from pathlib import Path

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
# Purpose:
# This cell loads the raw dataset and calculates missing values.
# We use this information to identify which features require investigation.
# No preprocessing is performed here.




RAW_DATA = Path("../data/raw/cs-training.csv")

df = pd.read_csv(RAW_DATA)

print("Dataset shape:")
print(df.shape)


missing_summary = (
    df.isnull()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] 
    / len(df)
    * 100
)


missing_summary = (
    missing_summary
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
)


missing_summary

## MonthlyIncome Missingness Analysis

MonthlyIncome contains 29,731 missing values (19.82%).

Based on domain understanding and dataset characteristics, the missingness is unlikely to be MCAR because income reporting behavior may depend on borrower characteristics.

The missingness is assumed to be primarily MAR, with possible MNAR behavior.

Because income is an important financial variable and missingness itself may contain information, we will preserve the missing pattern using a missing indicator feature.

Candidate approaches:
- Median imputation
- KNN imputation
- Regression imputation
- MICE

The final method will be selected after validation comparison.

## NumberOfDependents Missingness Analysis

NumberOfDependents contains 3,924 missing values (2.62%).

Because the missing percentage is low and the feature is demographic rather than financial, the missingness is likely MCAR or weak MAR.

Removing rows would unnecessarily reduce training data.

Candidate approaches:
- Median imputation
- Mode imputation
- Constant value imputation

The final method will be selected after validation.

In [ ]:
# Purpose:
# Check whether rows with missing values have different default rates.
# If missing rows have different target behavior,
# missingness itself contains predictive information.

missing_analysis = pd.DataFrame()

for col in ["MonthlyIncome", "NumberOfDependents"]:
    
    missing_rate = (
        df.groupby(df[col].isna())["SeriousDlqin2yrs"]
        .mean()
        .to_frame("default_rate")
    )
    
    print("\nFeature:", col)
    print(missing_rate)

## Missingness Impact on Target

To determine whether missing values contain predictive information, we compared default rates between missing and non-missing groups.

### MonthlyIncome

Missing values have a default rate of 5.61% compared with 6.95% for observed values.

The difference suggests that missingness is not completely random. A missing indicator feature will be created to preserve this information.

### NumberOfDependents

Missing values have a default rate of 4.56% compared with 6.74% for observed values.

The difference indicates that missingness may contain predictive information.

### Final Decision

For both features:

- Create missing indicator features.
- Apply imputation to replace missing values.
- Preserve the original missing pattern.

Final preprocessing will compare imputation methods using validation performance.

In [ ]:
# Purpose:
# Compare statistical characteristics of missing and non-missing groups.
# Helps determine whether missing rows come from a different population.

for col in ["MonthlyIncome", "NumberOfDependents"]:
    
    print("\n========================")
    print(col)
    
    print(
        df.groupby(df[col].isna())[col]
        .describe()
    )

## Distribution Analysis of Missing Groups

Direct statistical comparison of the missing feature itself is not possible because missing values have no observed distribution.

Instead, we compare borrowers with missing values against borrowers with observed values using other available characteristics.

This helps determine whether missingness occurs randomly or is associated with borrower profiles.

In [ ]:
# Purpose:
# Compare borrower characteristics between missing and non-missing groups.
# This helps identify whether missingness depends on other variables (MAR).

features_to_compare = [
    "age",
    "DebtRatio",
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "SeriousDlqin2yrs"
]


for missing_feature in [
    "MonthlyIncome",
    "NumberOfDependents"
]:
    
    print("\n==============================")
    print("Missing feature:", missing_feature)
    
    comparison = (
        df.groupby(df[missing_feature].isna())[features_to_compare]
        .mean()
        .T
    )
    
    comparison.columns = [
        "Observed",
        "Missing"
    ]
    
    print(comparison)

## Missingness Mechanism Conclusion

The analysis shows that both missing features have different borrower profiles compared with observed values.

Missing groups differ in:

- age
- debt ratio
- credit utilization
- delinquency history

Therefore, missingness is unlikely to be MCAR.

The most reasonable assumption is:

- MonthlyIncome → MAR (possible MNAR behavior)
- NumberOfDependents → MAR

Because missingness contains information, missing indicator features will be preserved.

Next, different imputation methods will be evaluated using validation performance.

In [ ]:
# Purpose:
# Create temporary missing indicators.
# These features preserve information about whether values were originally missing.
# We will evaluate their usefulness before finalizing the pipeline.

for col in ["MonthlyIncome", "NumberOfDependents"]:
    df[f"{col}_missing"] = df[col].isna().astype(int)

df[
    [
        "MonthlyIncome_missing",
        "NumberOfDependents_missing"
    ]
].value_counts()

## Missing Indicator Feature Decision

Missing indicator features were created to preserve information contained in missingness patterns.

The analysis showed:

- 25,807 records have missing MonthlyIncome only.
- 3,924 records have both MonthlyIncome and NumberOfDependents missing.
- No records have NumberOfDependents missing without MonthlyIncome missing.

This indicates that missing values are not randomly distributed across features and may represent a specific borrower profile or data collection pattern.

Decision:

Keep missing indicator features:

- MonthlyIncome_missing
- NumberOfDependents_missing

These features will be included during model training.

## Imputation Strategy Selection

After understanding the missingness mechanism, the next step is selecting an imputation method.

The choice depends on:

- Missing percentage
- Feature distribution
- Feature meaning
- Relationship with other variables
- Model performance after preprocessing

### MonthlyIncome

Characteristics:

- 19.82% missing
- Continuous numerical feature
- Highly right-skewed distribution
- Extreme values exist
- Important financial variable

Candidate methods:

### Median Imputation

Advantages:
- Robust to skewness and outliers
- Simple and production friendly

Disadvantage:
- Does not use relationships between features

---

### KNN Imputation

Advantages:
- Uses similarity between borrowers
- Can capture relationships between financial features

Disadvantages:
- Requires scaling
- Computationally expensive
- Sensitive to distance metrics

---

### MICE / Iterative Imputation

Advantages:
- Models each missing feature using other variables
- Can capture complex relationships

Disadvantages:
- More computationally expensive
- More complex for production pipelines

---

### NumberOfDependents

Characteristics:

- 2.62% missing
- Count-based feature
- Majority of values are zero

Candidate methods:

- Median imputation
- Mode imputation

Because the missing percentage is low, simple methods are expected to perform well.

Final selection will be based on validation performance.

In [ ]:
# Purpose:
# Create the first interim dataset after missing value handling.
# This version uses median imputation as a baseline.
# Raw data is not modified.
# Output will be stored in data/interim for DVC tracking.


RAW_DATA = Path("../data/raw/cs-training.csv")

INTERIM_DATA = Path(
    "../data/interim/missing_values_median_v1.csv"
)


df = pd.read_csv(RAW_DATA)


# Create missing indicators before imputation
df["MonthlyIncome_missing"] = (
    df["MonthlyIncome"].isna()
    .astype(int)
)

df["NumberOfDependents_missing"] = (
    df["NumberOfDependents"].isna()
    .astype(int)
)


# Median imputation
df["MonthlyIncome"] = (
    df["MonthlyIncome"]
    .fillna(df["MonthlyIncome"].median())
)

df["NumberOfDependents"] = (
    df["NumberOfDependents"]
    .fillna(df["NumberOfDependents"].median())
)


# Verify missing values are removed
print(
    df[
        [
            "MonthlyIncome",
            "NumberOfDependents"
        ]
    ]
    .isnull()
    .sum()
)


# Save interim dataset
INTERIM_DATA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    INTERIM_DATA,
    index=False
)


print("Saved:", INTERIM_DATA)
print("Shape:", df.shape)

## KNN Imputation Experiment

K-Nearest Neighbors (KNN) imputation estimates missing values by finding similar observations.

For a missing value:

1. Calculate distance between samples using available features.
2. Select the k most similar samples.
3. Replace the missing value using the neighbors' values.

Advantages:

- Uses relationships between features.
- Can capture borrower similarity.

Disadvantages:

- Sensitive to feature scale.
- Computationally expensive for large datasets.
- Distance can be affected by outliers.

Because this dataset contains financial variables with different scales, scaling is required before KNN imputation.

This experiment will compare KNN imputation against the median baseline.

In [ ]:
# Purpose:
# Create an interim dataset using KNN imputation.
# This is an experiment version and will be compared against median imputation.
# Raw data is not modified.



RAW_DATA = Path("../data/raw/cs-training.csv")

INTERIM_DATA = Path(
    "../data/interim/missing_values_knn_v1.csv"
)


df = pd.read_csv(RAW_DATA)


# Create missing indicators
df["MonthlyIncome_missing"] = (
    df["MonthlyIncome"].isna()
    .astype(int)
)

df["NumberOfDependents_missing"] = (
    df["NumberOfDependents"].isna()
    .astype(int)
)


# Keep original columns
columns = df.columns


# Scaling before KNN
scaler = StandardScaler()

scaled_data = scaler.fit_transform(df)


# KNN imputation
imputer = KNNImputer(
    n_neighbors=5
)

imputed_data = imputer.fit_transform(
    scaled_data
)


# Convert back to dataframe
df_knn = pd.DataFrame(
    imputed_data,
    columns=columns
)


# Reverse scaling
df_knn = pd.DataFrame(
    scaler.inverse_transform(df_knn),
    columns=columns
)


# Verify missing values
print(
    df_knn.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head()
)


# Save experiment dataset
INTERIM_DATA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_knn.to_csv(
    INTERIM_DATA,
    index=False
)


print("Saved:", INTERIM_DATA)
print("Shape:", df_knn.shape)

## KNN Imputation Experiment Result

KNN imputation was successfully applied after feature scaling.

The resulting dataset:

- Contains no missing values.
- Preserves all original samples.
- Maintains missing indicator features.

However, successful imputation does not guarantee better predictive performance.

The KNN version will be evaluated against the median imputation baseline during model validation.

The final imputation method will be selected based on model performance and production considerations.

## MICE / Iterative Imputation Experiment

Iterative imputation estimates missing values by modeling each feature with missing values as a prediction problem.

The process:

1. Select a feature with missing values.
2. Use other features to predict the missing values.
3. Repeat the process iteratively until estimates stabilize.

For this project:

MonthlyIncome may depend on:

- age
- DebtRatio
- credit utilization
- repayment history
- number of dependents

Therefore, a multivariate approach may capture relationships better than simple median imputation.

Advantages:

- Uses relationships between variables.
- Suitable when missingness is MAR.
- Can preserve feature relationships.

Disadvantages:

- More computationally expensive.
- More complex than median imputation.
- Requires careful validation.

The final decision will depend on predictive performance and production requirements.

In [ ]:
# Purpose:
# Create an interim dataset using Iterative Imputation (MICE).
# This experiment estimates missing values using relationships between features.
# Raw data remains unchanged.

RAW_DATA = Path("../data/raw/cs-training.csv")

INTERIM_DATA = Path(
    "../data/interim/missing_values_mice_v1.csv"
)


df = pd.read_csv(RAW_DATA)


# Create missing indicators before imputation
df["MonthlyIncome_missing"] = (
    df["MonthlyIncome"]
    .isna()
    .astype(int)
)

df["NumberOfDependents_missing"] = (
    df["NumberOfDependents"]
    .isna()
    .astype(int)
)


# Initialize MICE
imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)


# Apply imputation
df_mice = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)


# Check missing values
print(
    df_mice.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head()
)


# Save interim dataset
INTERIM_DATA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_mice.to_csv(
    INTERIM_DATA,
    index=False
)


print("Saved:", INTERIM_DATA)
print("Shape:", df_mice.shape)

# Comparing Missing Value Imputation Strategies

Three imputation approaches were tested:

1. Median Imputation
2. KNN Imputation
3. MICE / Iterative Imputation

To select the final approach, we compare their impact on a baseline machine learning model.

The comparison keeps:

- Same train/test split
- Same model
- Same evaluation metrics

Only the preprocessing method changes.

The selected method should balance:

- Predictive performance
- Computational cost
- Production simplicity

In [ ]:
# Purpose:
# Load all preprocessing experiment datasets.
# Each dataset represents a different missing-value strategy.

DATA_PATH = Path("../data/interim")


median_df = pd.read_csv(
    DATA_PATH / "missing_values_median_v1.csv"
)

knn_df = pd.read_csv(
    DATA_PATH / "missing_values_knn_v1.csv"
)

mice_df = pd.read_csv(
    DATA_PATH / "missing_values_mice_v1.csv"
)


print("Median:", median_df.shape)
print("KNN:", knn_df.shape)
print("MICE:", mice_df.shape)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Purpose:
# Train the same baseline model on each preprocessing version.
# This isolates the effect of missing value handling.


def evaluate_dataset(data, name):

    # Remove index column if present
    if "Unnamed: 0" in data.columns:
        data = data.drop(
            columns=["Unnamed: 0"]
        )

    X = data.drop(
        columns=["SeriousDlqin2yrs"]
    )

    y = data["SeriousDlqin2yrs"]


    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    probabilities = model.predict_proba(
        X_test
    )[:,1]


    predictions = (
        probabilities >= 0.5
    ).astype(int)


    return {
        "Method": name,
        "Precision": precision_score(
            y_test,
            predictions
        ),
        "Recall": recall_score(
            y_test,
            predictions
        ),
        "F1": f1_score(
            y_test,
            predictions
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    }



results = []


results.append(
    evaluate_dataset(
        median_df,
        "Median"
    )
)

results.append(
    evaluate_dataset(
        knn_df,
        "KNN"
    )
)

results.append(
    evaluate_dataset(
        mice_df,
        "MICE"
    )
)


results_df = pd.DataFrame(results)

results_df

# Missing Values Final Decision

Three imputation strategies were evaluated:

- Median imputation
- KNN imputation
- MICE / Iterative imputation

The evaluation was performed using the same baseline Logistic Regression model.

Results showed:

- MICE achieved the highest ROC-AUC (0.8063) and F1-score (0.3241).
- Median imputation achieved the highest recall (0.6489).
- KNN did not provide a significant improvement over simpler methods.

Final decisions:

### MonthlyIncome

Selected:
- MICE / Iterative Imputation
- MonthlyIncome_missing indicator

Reason:
- High missing percentage.
- Missingness is likely MAR.
- Feature is important and related to other borrower characteristics.

### NumberOfDependents

Selected:
- Median imputation
- NumberOfDependents_missing indicator

Reason:
- Low missing percentage.
- Count-based feature.
- Simple robust method is sufficient.

All preprocessing experiments were stored separately in the interim data folder and tracked with DVC.

# Step 2.2 — Invalid Values Analysis

Invalid values are observations that violate the meaning or possible range of a feature.

They are different from outliers.

## Outlier

A value can be extreme but valid.

Example:
- A borrower with a very high income.

## Invalid Value

A value that cannot logically exist.

Example:
- Negative age.
- Negative number of dependents.

Invalid values can occur because of:

- Data entry mistakes
- Measurement errors
- Encoding problems

Before modifying any value, we analyze:

1. Domain constraints.
2. Feature meaning.
3. Distribution from EDA.

Only impossible values will be corrected or removed.

In [ ]:
# Purpose:
# Inspect minimum and maximum values of all numerical features.
# This identifies possible invalid values based on domain knowledge.

RAW_DATA = Path("../data/raw/cs-training.csv")

df = pd.read_csv(RAW_DATA)


numeric_summary = (
    df.describe()
    .T[
        [
            "min",
            "max",
            "mean",
            "50%"
        ]
    ]
)


numeric_summary

## Invalid Value Investigation

Initial range analysis identified possible suspicious values:

- Age = 0 or very high values
- Extremely high DebtRatio values
- Extremely high revolving utilization values
- High delinquency counts

These values will not be removed automatically.

The next step is to measure their frequency and determine whether they represent:
- data errors,
- valid extreme borrowers,
- or outliers requiring later treatment.

In [ ]:
# Purpose:
# Count suspicious values identified during range analysis.
# Frequency helps decide whether values are invalid or rare but valid.

checks = {
    "age_zero": df["age"].eq(0).sum(),
    "age_above_100": (df["age"] > 100).sum(),

    "DebtRatio_above_100": (
        df["DebtRatio"] > 100
    ).sum(),

    "Utilization_above_100": (
        df["RevolvingUtilizationOfUnsecuredLines"] > 100
    ).sum(),

    "DaysLate_98": (
        df["NumberOfTimes90DaysLate"] == 98
    ).sum(),

    "30_59_days_98": (
        df["NumberOfTime30-59DaysPastDueNotWorse"] == 98
    ).sum(),

    "60_89_days_98": (
        df["NumberOfTime60-89DaysPastDueNotWorse"] == 98
    ).sum()
}


pd.Series(checks)

# Invalid Value Investigation Results

Range analysis identified several suspicious values.

## Age

Age values of 0 and values above 100 are considered invalid because they violate borrower domain assumptions.

Decision:
- Convert these values to missing values.
- Handle through imputation.

## DebtRatio

Although many values exceed 100, these observations represent a significant portion of the dataset.

They may represent borrowers with extreme debt relative to income.

Decision:
- Keep values.
- Handle during outlier analysis.

## RevolvingUtilizationOfUnsecuredLines

Values above 100 require further investigation because utilization can exceed 1 but extremely large values may indicate data issues.

## Delinquency Features

The value 98 appears consistently across multiple delinquency variables.

This suggests a possible encoding issue and requires further investigation before treatment.

In [ ]:
# Purpose:
# Investigate rows where delinquency variables contain value 98.
# Determine whether 98 represents a real value or an encoding artifact.

late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


df[
    df[late_cols].eq(98).any(axis=1)
][late_cols + ["SeriousDlqin2yrs"]].head(20)

## Delinquency Feature Encoding Investigation

The value 98 was found in three delinquency variables:

- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTimes90DaysLate

All occurrences appeared simultaneously across the three features.

Because a borrower having exactly 98 occurrences across multiple delinquency periods is unrealistic, this value is considered an encoded missing value rather than a valid observation.

Decision:

Replace value 98 with missing values (NaN).

These newly created missing values will be handled during the missing value preprocessing stage.

In [ ]:
# Purpose:
# Inspect extreme revolving utilization values.
# Determine whether very large values are valid or data errors.

df[
    df["RevolvingUtilizationOfUnsecuredLines"] > 100
][
    [
        "RevolvingUtilizationOfUnsecuredLines",
        "age",
        "DebtRatio",
        "MonthlyIncome",
        "SeriousDlqin2yrs"
    ]
].sort_values(
    by="RevolvingUtilizationOfUnsecuredLines",
    ascending=False
).head(20)

## Revolving Utilization Invalid Value Investigation

RevolvingUtilizationOfUnsecuredLines contained 223 observations above 100.

Although utilization values slightly above 1 can be valid because balances may exceed credit limits, extremely large values such as 50,708 are not realistic.

These observations represent only 0.15% of the dataset and are several orders of magnitude larger than normal utilization values.

Decision:

Values greater than 100 will be treated as invalid and converted to missing values.

The missing values will be handled during preprocessing using the selected imputation strategy.

In [ ]:
# Purpose:
# Create an interim dataset after handling confirmed invalid values.
# Raw data is preserved.
# This version will be tracked with DVC.



RAW_DATA = Path("../data/raw/cs-training.csv")

INTERIM_DATA = Path(
    "../data/interim/invalid_values_v1.csv"
)


df = pd.read_csv(RAW_DATA)


# Age invalid values
df.loc[
    (df["age"] == 0) | (df["age"] > 100),
    "age"
] = None


# Revolving utilization impossible values
df.loc[
    df["RevolvingUtilizationOfUnsecuredLines"] > 100,
    "RevolvingUtilizationOfUnsecuredLines"
] = None


# Delinquency encoded missing value
late_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


for col in late_columns:
    df.loc[
        df[col] == 98,
        col
    ] = None


# Save interim dataset

INTERIM_DATA.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    INTERIM_DATA,
    index=False
)


print("Saved:", INTERIM_DATA)
print("Shape:", df.shape)


# Check changes
print("\nNew missing values:")
print(
    df.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
)

# Invalid Values Final Decision

Invalid values were investigated using domain knowledge and frequency analysis.

The following treatments were applied:

## Age

Values equal to 0 and greater than 100 were considered impossible for a credit borrower.

Action:
- Converted to missing values.

## RevolvingUtilizationOfUnsecuredLines

Extremely large values above 100 were considered unrealistic because they represent impossible utilization percentages.

Action:
- Converted to missing values.

## Delinquency Variables

The value 98 appeared simultaneously across multiple delinquency features.

This pattern indicated an encoded missing value.

Action:
- Replaced 98 with missing values.

## DebtRatio

Although extreme values existed, they represented a significant portion of the dataset and may contain useful risk information.

Action:
- Preserved for later outlier analysis.

No rows were removed during invalid value handling.

The cleaned interim dataset was stored and versioned using DVC.

# Step 2.3 — Outlier Treatment

Outliers are observations that are significantly different from the majority of the data distribution.

An outlier is not always an error.

Examples:

- A very high income borrower may be valid.
- A borrower with extremely high debt may represent a real high-risk case.

Before treatment, we analyze:

- Distribution shape
- Feature meaning
- Percentage of extreme observations
- Impact on model performance

Possible treatments:

## No Treatment

Keep original values when extreme values contain useful information.

## Winsorization / Clipping

Limit extreme values to a selected percentile.

Example:

Values above the 99th percentile are replaced by the 99th percentile.

## Log Transformation

Used for highly right-skewed positive numerical features.

It reduces the impact of very large values while preserving ordering.

## Robust Scaling

Uses median and IQR instead of mean and standard deviation.

Useful when extreme values exist.

The final method is selected based on feature behavior and model performance.

In [ ]:
# Purpose:
# Analyze distribution characteristics of numerical features.
# Used to decide whether outlier treatment is required.


DATA_PATH = Path("../data/interim/invalid_values_v1.csv")

df = pd.read_csv(DATA_PATH)


numeric_features = df.select_dtypes(
    include=["int64", "float64"]
).columns


outlier_profile = pd.DataFrame(
    index=numeric_features
)


outlier_profile["skewness"] = (
    df[numeric_features]
    .skew()
)


outlier_profile["q1"] = (
    df[numeric_features]
    .quantile(0.25)
)


outlier_profile["median"] = (
    df[numeric_features]
    .median()
)


outlier_profile["q3"] = (
    df[numeric_features]
    .quantile(0.75)
)


outlier_profile["max"] = (
    df[numeric_features]
    .max()
)


outlier_profile.sort_values(
    by="skewness",
    ascending=False
)

In [ ]:
# Purpose:
# Analyze frequency distribution of delinquency count features.
# Determine whether extreme values are rare errors or meaningful risk signals.

late_features = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


for col in late_features:
    print("\n====================")
    print(col)
    print(
        df[col]
        .value_counts()
        .sort_index()
        .tail(15)
    )

In [ ]:
# Purpose:
# Investigate whether value 96 is another encoded invalid value
# in delinquency count features.

late_features = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


for col in late_features:
    print("\n====================")
    print(col)

    print(
        "Value 96 count:",
        (df[col] == 96).sum()
    )

    print(
        df[
            df[col] == 96
        ][
            [
                col,
                "SeriousDlqin2yrs"
            ]
        ].head(10)
    )

In [ ]:
# Purpose:
# Create updated invalid-value cleaning version.
# Adds handling for delinquency encoded values 96 and 98.
# Previous v1 dataset remains unchanged for reproducibility.

INVALID_DATA = Path(
    "../data/interim/invalid_values_v1.csv"
)

UPDATED_DATA = Path(
    "../data/interim/invalid_values_v2.csv"
)


df_invalid = pd.read_csv(
    INVALID_DATA
)


# Replace encoded invalid delinquency values

late_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


for col in late_columns:
    df_invalid.loc[
        df_invalid[col].isin([96, 98]),
        col
    ] = None


df_invalid.to_csv(
    UPDATED_DATA,
    index=False
)


print("Saved:", UPDATED_DATA)

print(
    df_invalid[late_columns]
    .isnull()
    .sum()
)

# MonthlyIncome Outlier Treatment Decision

MonthlyIncome showed extreme right skew:

- Skewness: 114
- Large difference between median and maximum value.

The feature represents income, where high values can be legitimate.

Removing high-income borrowers would remove real information.

Decision:

Apply log1p transformation.

Reason:

- Preserves all observations.
- Reduces extreme scale differences.
- Suitable for positive financial variables.

Transformation:

log1p(x) = log(1 + x)

In [ ]:
# Purpose:
# Create an experiment version with MonthlyIncome log transformation.
# Original values are preserved in previous DVC versions.

OUTLIER_DATA = Path(
    "../data/interim/invalid_values_v2.csv"
)

df_outlier = pd.read_csv(
    OUTLIER_DATA
)


df_outlier["MonthlyIncome_log"] = (
    np.log1p(
        df_outlier["MonthlyIncome"]
    )
)


df_outlier[
    [
        "MonthlyIncome",
        "MonthlyIncome_log"
    ]
].describe()

# MonthlyIncome Outlier Treatment Result

MonthlyIncome showed extreme right skew with:

- Skewness: 114
- Maximum value much larger than the median.

Removing high-income observations was rejected because extreme income values can represent legitimate borrowers.

A log1p transformation was applied.

Result:

- Reduced skewness impact.
- Compressed extreme values.
- Preserved all observations.

Decision:

Use MonthlyIncome_log as the transformed representation for modeling.

In [ ]:
# Purpose:
# Analyze RevolvingUtilization distribution after invalid value removal.

df_outlier[
    "RevolvingUtilizationOfUnsecuredLines"
].describe()

# Revolving Utilization Outlier Treatment Decision

After removing invalid values above 100, the feature still showed strong right skew.

Evidence:

- Median: 0.154
- Maximum: 95
- Large difference between central values and extreme values.

Because utilization values above 1 can be valid, removing high values was rejected.

Decision:

Apply log1p transformation.

Reason:

- Preserves observations.
- Compresses extreme values.
- Reduces the impact of heavy right skew.

In [ ]:
# Purpose:
# Apply log1p transformation to revolving utilization.
# Create transformed feature for later model comparison.

df_outlier["RevolvingUtilization_log"] = (
    np.log1p(
        df_outlier[
            "RevolvingUtilizationOfUnsecuredLines"
        ]
    )
)


df_outlier[
    [
        "RevolvingUtilizationOfUnsecuredLines",
        "RevolvingUtilization_log"
    ]
].describe()

# Revolving Utilization Outlier Treatment Result

RevolvingUtilizationOfUnsecuredLines remained highly right-skewed after invalid value removal.

Evidence:

- Median: 0.154
- Maximum: 95
- Standard deviation: 0.689

Values above 1 were not removed because high utilization can represent real credit behavior.

A log1p transformation was applied.

Result:

- Reduced distribution spread.
- Compressed extreme values.
- Preserved all observations.

Decision:

Use RevolvingUtilization_log for modeling.

In [ ]:
# Purpose:
# Inspect the highest DebtRatio values.
# Determine whether extreme values are invalid or meaningful.

df_outlier[
    [
        "DebtRatio",
        "MonthlyIncome",
        "age",
        "SeriousDlqin2yrs"
    ]
].sort_values(
    by="DebtRatio",
    ascending=False
).head(20)

# DebtRatio Outlier Treatment Decision

DebtRatio showed extreme right skew:

- Skewness: 95
- Median: 0.366
- Maximum: 329664

Investigation showed that the highest values were strongly associated with missing MonthlyIncome values.

These observations were not removed because they represent a large portion of the dataset and may contain information.

A log1p transformation was selected.

Reason:

- Preserves all observations.
- Reduces extreme scale differences.
- Suitable for highly skewed positive ratio features.

Decision:

Use DebtRatio_log for modeling.

In [ ]:
# Purpose:
# Apply log transformation to DebtRatio.
# Reduce extreme scale while preserving observations.

df_outlier["DebtRatio_log"] = (
    np.log1p(
        df_outlier["DebtRatio"]
    )
)


df_outlier[
    [
        "DebtRatio",
        "DebtRatio_log"
    ]
].describe()

# DebtRatio Outlier Treatment Result

DebtRatio showed extreme right skew:

- Skewness: 95
- Median: 0.366
- Maximum: 329664

Investigation showed that extreme values were often associated with missing MonthlyIncome values.

Removing these observations was rejected because they represent a large dataset segment.

A log1p transformation was applied.

Result:

- Reduced extreme scale.
- Preserved all observations.
- Maintained borrower ordering.

Decision:

Create DebtRatio_log for modeling.

In [ ]:
# Purpose:
# Save the preprocessing experiment after outlier treatment.
# This creates a new interim dataset version for DVC tracking.

OUTLIER_VERSION = Path(
    "../data/interim/outlier_treatment_v1.csv"
)


# Keep transformed features
df_outlier.to_csv(
    OUTLIER_VERSION,
    index=False
)


print("Saved:", OUTLIER_VERSION)
print("Shape:", df_outlier.shape)

# Step 2.5 — Feature Scaling

Feature scaling changes the numerical range of features so that machine learning algorithms can process them effectively.

Different algorithms react differently to feature scale.

## Algorithms sensitive to scaling

Examples:

- Logistic Regression
- K-Nearest Neighbors
- Support Vector Machines
- Neural Networks

These algorithms use distances or optimization based on feature values.

Example:

A feature ranging from:

1000 - 100000

can dominate a feature ranging from:

0 - 1.

## Algorithms usually not sensitive to scaling

Examples:

- Decision Trees
- Random Forest
- XGBoost
- LightGBM
- CatBoost

Tree algorithms split values based on thresholds and are mostly unaffected by feature magnitude.

## Scaling Methods

### StandardScaler

Transforms data to:

- mean = 0
- standard deviation = 1

Suitable for approximately normal features.

### MinMaxScaler

Transforms values into:

0 to 1 range.

Sensitive to outliers.

### RobustScaler

Uses:

- median
- interquartile range (IQR)

More resistant to extreme values.

## Decision

Scaling will be evaluated based on:

- model requirements
- feature distributions
- presence of outliers

Tree-based models will use unscaled features.
Linear baseline models may use scaled features.

In [ ]:
# Purpose:
# Inspect feature ranges after previous preprocessing steps.
# Used to decide whether scaling is required.

SCALING_DATA = Path(
    "../data/interim/outlier_treatment_v1.csv"
)

df_scale = pd.read_csv(
    SCALING_DATA
)


scale_summary = (
    df_scale
    .describe()
    .T[
        [
            "min",
            "max",
            "mean",
            "50%"
        ]
    ]
)


scale_summary

# Feature Scaling Decision

Feature scaling was evaluated based on algorithm requirements.

Tree-based models:

- Random Forest
- XGBoost
- LightGBM
- CatBoost

do not require scaling because they split data using thresholds.

Logistic Regression is sensitive to feature magnitude because optimization depends on feature values.

Decision:

- Apply RobustScaler for Logistic Regression.
- Keep original numerical features for tree-based models.

RobustScaler was selected instead of StandardScaler because the dataset still contains skewed financial variables and extreme observations.

In [ ]:
# Purpose:
# Create scaled feature matrix for Logistic Regression.
# Tree model datasets remain unscaled.

from sklearn.preprocessing import RobustScaler


# Remove target and index columns

X = df_scale.drop(
    columns=[
        "SeriousDlqin2yrs",
        "Unnamed: 0"
    ]
)


y = df_scale[
    "SeriousDlqin2yrs"
]


scaler = RobustScaler()


X_scaled = scaler.fit_transform(
    X
)


X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)


X_scaled.describe()

In [ ]:
# Purpose:
# Save RobustScaler transformed dataset for Logistic Regression experiments.
# This creates a separate reproducible preprocessing version.

SCALED_DATA = Path(
    "../data/interim/scaled_logistic_v1.csv"
)


X_scaled["SeriousDlqin2yrs"] = y.values


X_scaled.to_csv(
    SCALED_DATA,
    index=False
)


print("Saved:", SCALED_DATA)
print("Shape:", X_scaled.shape)

# Step 2.7 — Class Imbalance Handling

Class imbalance occurs when the target classes are not represented equally.

In credit risk prediction, this is common because default cases are usually much fewer than non-default cases.

A model trained on imbalanced data may achieve high accuracy by predicting the majority class while failing to detect important minority cases.

Example:

If 93% of customers are non-default and 7% are default:

A model predicting "non-default" for every customer gets 93% accuracy but has zero ability to detect risky customers.

For this project:

Target:

SeriousDlqin2yrs

Meaning:

0 → No serious delinquency
1 → Serious delinquency

Primary objective:

Detect as many defaults as possible.

Therefore:

Primary metric:
- Recall

Constraint:
- Maintain acceptable precision

Possible approaches:

## 1. No Handling

Train on original distribution.

Used as baseline.

## 2. Class Weights

Increase penalty for misclassifying minority class.

Example:

A default error receives higher importance.

Supported by:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM
- CatBoost

## 3. SMOTE

Synthetic Minority Oversampling Technique.

Creates synthetic minority samples instead of duplicating existing ones.

## 4. SMOTE-Tomek

Combines:

SMOTE:
- creates minority samples

Tomek links:
- removes overlapping samples

## 5. SMOTE-ENN

Combines:

SMOTE:
- oversampling

ENN:
- removes noisy samples

Decision will be based on validation performance, not assumptions.

In [ ]:
# Purpose:
# Measure original target class distribution.
# Establish baseline imbalance before applying techniques.

IMBALANCE_DATA = Path(
    "../data/interim/outlier_treatment_v1.csv"
)

df_imbalance = pd.read_csv(
    IMBALANCE_DATA
)


class_distribution = (
    df_imbalance["SeriousDlqin2yrs"]
    .value_counts()
)


class_percentage = (
    df_imbalance["SeriousDlqin2yrs"]
    .value_counts(normalize=True)
    * 100
)


print("Class Count:")
print(class_distribution)

print("\nClass Percentage:")
print(class_percentage)

In [ ]:
# Purpose:
# Create reproducible train-validation split.
# Stratification preserves class distribution in both sets.

TARGET = "SeriousDlqin2yrs"


X = df_imbalance.drop(
    columns=[TARGET, "Unnamed: 0"]
)


y = df_imbalance[TARGET]


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)


print("\nTrain distribution:")
print(y_train.value_counts(normalize=True))


print("\nValidation distribution:")
print(y_valid.value_counts(normalize=True))

# Class Imbalance Baseline Model

Before applying imbalance handling techniques, a baseline Logistic Regression model is trained on the original imbalanced dataset.

Purpose:

- Establish reference performance.
- Understand how the model behaves with the natural class distribution.
- Compare improvements from class weights and resampling methods.

The validation dataset remains untouched for all experiments.

Metrics:

- Precision
- Recall
- F1-score
- ROC-AUC

Recall is prioritized because the objective is to identify as many default cases as possible.

# Final Missing Value Imputation Decision

After invalid value correction and outlier treatment, missing values must be handled before model training.

Decisions:

## MonthlyIncome

Missingness:
- 19.8%

Previous experiments showed MICE achieved the best validation performance.

Decision:
- Use MICE imputation.

## NumberOfDependents

Missingness:
- 2.6%

The feature is a zero-heavy count variable.

Decision:
- Use median imputation.

## Age

Invalid values were converted to missing.

Decision:
- Use median imputation because age is continuous and the missing percentage is very small.

## RevolvingUtilizationOfUnsecuredLines

Invalid values were converted to missing.

Decision:
- Use median imputation because utilization is highly skewed.

## Delinquency Count Features

Invalid encoded values were converted to missing.

Decision:
- Replace with 0 because these features represent counts of delinquency events.

The final imputed dataset will be stored as a new DVC version.

In [ ]:
# Purpose:
# Create final imputed dataset after invalid value handling
# and outlier treatment.

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


FINAL_DATA = Path(
    "../data/interim/outlier_treatment_v1.csv"
)

OUTPUT_DATA = Path(
    "../data/interim/final_imputed_v1.csv"
)


df_final = pd.read_csv(
    FINAL_DATA
)


# MonthlyIncome MICE imputation

mice_imputer = IterativeImputer(
    random_state=42,
    max_iter=10
)


df_final["MonthlyIncome"] = (
    mice_imputer.fit_transform(
        df_final[
            ["MonthlyIncome"]
        ]
    )
)


# Median imputation

median_columns = [
    "NumberOfDependents",
    "age",
    "RevolvingUtilizationOfUnsecuredLines"
]


for col in median_columns:
    df_final[col] = (
        df_final[col]
        .fillna(
            df_final[col].median()
        )
    )


# Count features: missing means zero events

count_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]


for col in count_columns:
    df_final[col] = (
        df_final[col]
        .fillna(0)
    )


df_final.to_csv(
    OUTPUT_DATA,
    index=False
)


print("Saved:", OUTPUT_DATA)

print("\nRemaining missing values:")
print(
    df_final.isnull()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
)

print("\nShape:", df_final.shape)

In [ ]:
# Purpose:
# Load final imputed dataset and create reproducible
# train-validation split for imbalance experiments.


from sklearn.model_selection import train_test_split


DATA_PATH = Path(
    "../data/interim/final_imputed_v1.csv"
)


df_model = pd.read_csv(
    DATA_PATH
)


TARGET = "SeriousDlqin2yrs"


X = df_model.drop(
    columns=[
        TARGET,
        "Unnamed: 0"
    ]
)


y = df_model[TARGET]


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)


print("\nTrain class distribution:")
print(
    y_train.value_counts(
        normalize=True
    )
)


print("\nValidation class distribution:")
print(
    y_valid.value_counts(
        normalize=True
    )
)

In [ ]:
# Purpose:
# Check exactly which columns still contain missing values.

df_model.isnull().sum().sort_values(ascending=False)

In [ ]:
# Purpose:
# Recreate transformed log features after final imputation.
# This removes NaN values created from earlier transformations.

import numpy as np


# Recreate log transformations using imputed values

df_model["MonthlyIncome_log"] = np.log1p(
    df_model["MonthlyIncome"]
)


df_model["RevolvingUtilization_log"] = np.log1p(
    df_model["RevolvingUtilizationOfUnsecuredLines"]
)


# Verify missing values

print("Remaining missing values:")
print(
    df_model.isnull()
    .sum()
    .sort_values(ascending=False)
    .head()
)


# Save new version

OUTPUT_PATH = Path(
    "../data/interim/final_imputed_v2.csv"
)


df_model.to_csv(
    OUTPUT_PATH,
    index=False
)


print("\nSaved:", OUTPUT_PATH)
print("Shape:", df_model.shape)

# Baseline Model Without Imbalance Handling

A baseline Logistic Regression model is trained using the natural class distribution.

Purpose:

- Establish reference performance.
- Measure the effect of imbalance handling techniques later.

No:

- class weights
- oversampling
- synthetic samples

are applied.

The validation set remains unchanged.

This baseline shows how the model behaves when minority defaults are only 6.68% of the dataset.

In [ ]:
# Purpose:
# Load final validated dataset and confirm
# it is ready for modeling.


DATA_PATH = Path(
    "../data/interim/final_imputed_v2.csv"
)


df_model = pd.read_csv(
    DATA_PATH
)


print("Shape:")
print(df_model.shape)


print("\nMissing values:")
print(
    df_model.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


print("\nTarget distribution:")
print(
    df_model["SeriousDlqin2yrs"]
    .value_counts(normalize=True)
)

In [ ]:
# Purpose:
# Create reproducible train-validation split
# after completing preprocessing.

from sklearn.model_selection import train_test_split


TARGET = "SeriousDlqin2yrs"


X = df_model.drop(
    columns=[
        TARGET,
        "Unnamed: 0"
    ]
)


y = df_model[TARGET]


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)


print("\nTrain distribution:")
print(
    y_train.value_counts(
        normalize=True
    )
)


print("\nValidation distribution:")
print(
    y_valid.value_counts(
        normalize=True
    )
)

In [ ]:
# Purpose:
# Train baseline Logistic Regression without
# imbalance handling techniques.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


baseline_model.fit(
    X_train,
    y_train
)


y_pred = baseline_model.predict(
    X_valid
)


y_prob = baseline_model.predict_proba(
    X_valid
)[:, 1]


baseline_result = pd.DataFrame(
    {
        "Method": ["Baseline"],
        "Precision": [
            precision_score(
                y_valid,
                y_pred
            )
        ],
        "Recall": [
            recall_score(
                y_valid,
                y_pred
            )
        ],
        "F1": [
            f1_score(
                y_valid,
                y_pred
            )
        ],
        "ROC-AUC": [
            roc_auc_score(
                y_valid,
                y_prob
            )
        ]
    }
)


baseline_result

# Class Weight Experiment

Class weighting modifies the loss function by assigning higher importance to minority class errors.

For this credit risk problem:

- Misclassifying a default customer is more costly.
- The model receives a larger penalty for class 1 errors.

Advantages:

- No synthetic data generation.
- Simple to deploy.
- Supported by many algorithms.

The validation dataset remains unchanged.

In [ ]:
# Purpose:
# Train Logistic Regression using balanced class weights.

weighted_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)


weighted_model.fit(
    X_train,
    y_train
)


y_pred = weighted_model.predict(
    X_valid
)


y_prob = weighted_model.predict_proba(
    X_valid
)[:, 1]


weighted_result = pd.DataFrame(
    {
        "Method": ["Class Weight"],
        "Precision": [
            precision_score(
                y_valid,
                y_pred
            )
        ],
        "Recall": [
            recall_score(
                y_valid,
                y_pred
            )
        ],
        "F1": [
            f1_score(
                y_valid,
                y_pred
            )
        ],
        "ROC-AUC": [
            roc_auc_score(
                y_valid,
                y_prob
            )
        ]
    }
)


weighted_result

# SMOTE Experiment

SMOTE (Synthetic Minority Oversampling Technique) creates synthetic examples for the minority class.

Instead of duplicating existing default cases, it generates new samples between similar minority observations.

Important:

SMOTE is applied only to the training data.

The validation dataset remains unchanged.

This prevents data leakage.

Expected effect:

- Increase minority class learning.
- Improve recall.
- Possible reduction in precision due to more aggressive predictions.

In [ ]:
# Purpose:
# Apply SMOTE only on training data and evaluate Logistic Regression.

from imblearn.over_sampling import SMOTE


smote = SMOTE(
    random_state=42
)


X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


print("Before SMOTE:")
print(y_train.value_counts())


print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

# Logistic Regression with SMOTE

The Logistic Regression model is trained on the SMOTE-balanced training dataset.

The validation set is unchanged.

Purpose:

- Compare synthetic oversampling against class weighting.
- Measure whether balancing the training distribution improves default detection.

Evaluation:

- Precision
- Recall
- F1-score
- ROC-AUC

In [ ]:
# Purpose:
# Train Logistic Regression after SMOTE oversampling.

smote_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


smote_model.fit(
    X_train_smote,
    y_train_smote
)


y_pred = smote_model.predict(
    X_valid
)


y_prob = smote_model.predict_proba(
    X_valid
)[:, 1]


smote_result = pd.DataFrame(
    {
        "Method": ["SMOTE"],
        "Precision": [
            precision_score(
                y_valid,
                y_pred
            )
        ],
        "Recall": [
            recall_score(
                y_valid,
                y_pred
            )
        ],
        "F1": [
            f1_score(
                y_valid,
                y_pred
            )
        ],
        "ROC-AUC": [
            roc_auc_score(
                y_valid,
                y_prob
            )
        ]
    }
)


smote_result

# SMOTE-Tomek Experiment

SMOTE-Tomek combines:

1. SMOTE:
   - Generates synthetic minority samples.

2. Tomek Links:
   - Removes ambiguous samples located near the class boundary.

Purpose:

Reduce overlap between default and non-default classes.

Expected effect:

- Cleaner training data.
- Potential improvement in precision.
- Possible small reduction in recall.

In [ ]:
# Purpose:
# Apply SMOTE-Tomek only on training data.

from imblearn.combine import SMOTETomek


smote_tomek = SMOTETomek(
    random_state=42
)


X_train_st, y_train_st = smote_tomek.fit_resample(
    X_train,
    y_train
)


print("Before SMOTE-Tomek:")
print(y_train.value_counts())


print("\nAfter SMOTE-Tomek:")
print(y_train_st.value_counts())

In [ ]:
# Purpose:
# Train Logistic Regression using SMOTE-Tomek
# balanced training data.

smote_tomek_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


smote_tomek_model.fit(
    X_train_st,
    y_train_st
)


y_pred = smote_tomek_model.predict(
    X_valid
)


y_prob = smote_tomek_model.predict_proba(
    X_valid
)[:, 1]


smote_tomek_result = pd.DataFrame(
    {
        "Method": ["SMOTE-Tomek"],
        "Precision": [
            precision_score(
                y_valid,
                y_pred
            )
        ],
        "Recall": [
            recall_score(
                y_valid,
                y_pred
            )
        ],
        "F1": [
            f1_score(
                y_valid,
                y_pred
            )
        ],
        "ROC-AUC": [
            roc_auc_score(
                y_valid,
                y_prob
            )
        ]
    }
)


smote_tomek_result

# SMOTE-ENN Experiment

SMOTE-ENN combines:

1. SMOTE:
   - Generates synthetic minority examples.

2. Edited Nearest Neighbours (ENN):
   - Removes samples that disagree with their nearest neighbours.

Purpose:

Create a cleaner training dataset by removing ambiguous samples.

Expected behavior:

- Better class separation.
- Possible precision improvement.
- Possible recall reduction.

In [ ]:
# Purpose:
# Apply SMOTE-ENN only on training data.

from imblearn.combine import SMOTEENN


smote_enn = SMOTEENN(
    random_state=42
)


X_train_se, y_train_se = smote_enn.fit_resample(
    X_train,
    y_train
)


print("Before SMOTE-ENN:")
print(y_train.value_counts())


print("\nAfter SMOTE-ENN:")
print(y_train_se.value_counts())

In [ ]:
# Purpose:
# Train Logistic Regression using SMOTE-ENN
# processed training data.

smote_enn_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


smote_enn_model.fit(
    X_train_se,
    y_train_se
)


y_pred = smote_enn_model.predict(
    X_valid
)


y_prob = smote_enn_model.predict_proba(
    X_valid
)[:, 1]


smote_enn_result = pd.DataFrame(
    {
        "Method": ["SMOTE-ENN"],
        "Precision": [
            precision_score(
                y_valid,
                y_pred
            )
        ],
        "Recall": [
            recall_score(
                y_valid,
                y_pred
            )
        ],
        "F1": [
            f1_score(
                y_valid,
                y_pred
            )
        ],
        "ROC-AUC": [
            roc_auc_score(
                y_valid,
                y_prob
            )
        ]
    }
)


smote_enn_result

# Class Imbalance Handling Decision

Several imbalance handling techniques were evaluated using Logistic Regression.

Methods tested:

- No imbalance handling
- Class weights
- SMOTE
- SMOTE-Tomek
- SMOTE-ENN

Results:

The original dataset showed severe imbalance:
- Class 0: 93.3%
- Class 1: 6.7%

Observations:

- Baseline achieved high precision but very low recall.
- Class weighting significantly improved default detection.
- SMOTE achieved the highest F1-score, providing the best precision-recall balance.
- SMOTE-Tomek did not improve over SMOTE.
- SMOTE-ENN achieved the highest recall but reduced precision and F1-score.

Decision:

For the current baseline modeling stage:

SMOTE is selected because it provides the best F1-score and the best balance between identifying defaults and limiting false positives.

Threshold tuning will be performed later because the business requirement prioritizes recall while maintaining a precision constraint.

# Phase 2 Deliverable — Clean Baseline Dataset

The final processed dataset is created after completing preprocessing decisions.

Included:

- Missing value imputation
- Invalid value correction
- Outlier treatment
- Selected log transformations

Not included:

- SMOTE
- Any oversampling technique

Reason:

Resampling methods are training-time operations only.

The production dataset must represent real observations.

The final dataset is version controlled using DVC for reproducibility.

In [ ]:
# Purpose:
# Create final clean baseline dataset for modeling.
# This dataset contains only real observations.



PROCESSED_PATH = Path(
    "../data/processed"
)


PROCESSED_PATH.mkdir(
    exist_ok=True
)


OUTPUT_FILE = (
    PROCESSED_PATH /
    "credit_risk_clean_v1.csv"
)


df_model.to_csv(
    OUTPUT_FILE,
    index=False
)


print("Saved:", OUTPUT_FILE)

print("Shape:")
print(df_model.shape)


print("\nMissing values:")
print(
    df_model.isnull()
    .sum()
    .sum()
)

# Phase 2 Deliverable — Credit Risk Clean Baseline Dataset v1

## Dataset Purpose

This dataset is the final clean baseline dataset produced after completing the preprocessing phase.

Location:

`data/processed/credit_risk_clean_v1.csv`

The dataset contains only real customer observations.

Shape:

- Rows: 150,000
- Features: 14 input features + target/index columns

---

# Preprocessing Methods Applied

## 1. Missing Value Handling

### MonthlyIncome

Missing percentage:

~19.8%

Method selected:

MICE (Iterative Imputation)

Reason:

MICE achieved the best validation ROC-AUC among tested imputation methods.

---

### NumberOfDependents

Missing percentage:

~2.6%

Method selected:

Median imputation

Reason:

The feature is a zero-heavy count variable, making median a stable choice.

---

### Age

Invalid values were converted to missing.

Method selected:

Median imputation

Reason:

Very small missing percentage and continuous numerical feature.

---

### RevolvingUtilizationOfUnsecuredLines

Invalid values were converted to missing.

Method selected:

Median imputation

Reason:

Highly skewed financial utilization feature.

---

### Delinquency Count Features

Features:

- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTime60-89DaysPastDueNotWorse
- NumberOfTimes90DaysLate

Method selected:

Replace missing values with 0.

Reason:

These represent event counts. Zero represents no recorded delinquency events.

---

# 2. Invalid Value Handling

Applied corrections:

- Age values outside realistic ranges converted to missing.
- DebtRatio extreme invalid values investigated.
- RevolvingUtilization impossible values investigated.
- Encoded delinquency values (96/98) treated as invalid and converted.

---

# 3. Outlier Treatment

Based on EDA:

Extreme financial values were investigated.

Transformations applied:

- Log transformation for highly skewed variables.

Created features:

- MonthlyIncome_log
- RevolvingUtilization_log
- DebtRatio_log

---

# 4. Skewness Treatment

Applied log transformation using:

`log1p()`

Reason:

Financial variables showed extreme right skewness.

Benefits:

- Reduced effect of extreme values.
- Improved distribution shape.

---

# 5. Feature Scaling

Scaling was evaluated separately.

Scaling is not permanently stored in this dataset because:

- Tree-based models do not require scaling.
- Scaling will be included inside model pipelines when required.

---

# 6. Class Imbalance Handling Decision

Original imbalance:

- Class 0: 93.3%
- Class 1: 6.7%

Methods tested:

- No handling
- Class weights
- SMOTE
- SMOTE-Tomek
- SMOTE-ENN


Selected method:

SMOTE

Reason:

SMOTE achieved the highest F1-score:

F1 = 0.360449

Class weighting achieved the highest recall, but SMOTE provided the best precision-recall balance.

Important:

SMOTE is NOT included in this dataset.

It will only be applied to training data during model training.

---

# Final Dataset Rules

Included:

✓ Clean real customer data  
✓ Missing values handled  
✓ Invalid values handled  
✓ Selected feature transformations  

Not included:

✗ Synthetic SMOTE samples  
✗ Train-validation split  
✗ Model-specific scaling  

The dataset is version controlled using DVC to preserve reproducibility.